In [0]:
"""
03_gold_datamart.py

Purpose:
- Create gold_top_item datamart
- Calculate yearly views
- Rank items
- Determine most used platform
"""
from pyspark.sql.functions import (
    col,
    rank,
    count,
    row_number
)

from pyspark.sql.window import Window

# --------------------------------------------------
# Read Silver Tables
# --------------------------------------------------

event_df = spark.table("silver_event")

item_df = spark.table("silver_item")

In [0]:
# --------------------------------------------------
# Filter View Events
# --------------------------------------------------
# NOTE:
# Update event_type column if actual column name differs.

views_df = event_df.filter(
    col("payload_json.event_name") == "view_item"
)

views_df = views_df.select(

    "*",

    col("payload_json.event_name").alias("event_type"),

    col("payload_json.platform").alias("platform"),

    col("payload_json.parameter_value").alias("item_id")

)

views_df.show()
# --------------------------------------------------
# Total Item Views Per Year
# --------------------------------------------------

item_views  = (
    views_df
    .groupBy(
        "item_id",
        "year"
    )
    .agg(
        count("*").alias("total_views")
    )
)
print("Item Views:")
views_agg.show()

In [0]:
# --------------------------------------------------
# Rank Items by Views
# --------------------------------------------------

rank_window = Window.partitionBy(
    "year"
).orderBy(
    col("total_views").desc()
)

ranked_items = item_views.withColumn(
    "item_rank",
    rank().over(rank_window)
)

print("Ranked Items:")
ranked_df.show()

In [0]:

# --------------------------------------------------
# Calculate Most Used Platform
# --------------------------------------------------
# --------------------------------------------------
# Count Platform Usage
# --------------------------------------------------

platform_usage = (
    views_df
    .groupBy(
        "item_id",
        "year",
        "platform"
    )
    .agg(
        count("*").alias("platform_count")
    )
)

platform_window = Window.partitionBy(
    "item_id",
    "year"
).orderBy(
    col("platform_count").desc()
)

platform_ranked = (
    platform_usage
    .withColumn(
        "rn",
        row_number().over(platform_window)
    )
)
most_used_platform = (
    platform_ranked
    .filter(col("rn") == 1)
    .select(
        "item_id",
        "year",
        col("platform").alias("most_used_platform")
    )
)
# --------------------------------------------------
# Create Final top_item Datamart
# --------------------------------------------------

gold_top_item = ranked_items.join(
    most_used_platform,
    ["item_id", "year"],
    "left"
)
gold_top_item.show()




In [0]:
# --------------------------------------------------
# Save Gold Table
# --------------------------------------------------

gold_df.write.format("delta") \
    .partitionBy("year") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_top_item")

# --------------------------------------------------
# Validate Output
# --------------------------------------------------

gold_df.show()

print("Gold datamart created successfully.")